In [1]:
'''

This demo will use only the class 3, ie. the stock price after 5 years. However, it is possible to switch to 
the other classes by using the two variables class_to_demo and increase, as well as modyfing the number of tickers 
to test (100 out of 1470)
Seed = 123

The starting point is the file dataset_with_classes.csv, which contains the original data and the increase % for
each of the three classes:

- create the multiindex dataframe
- scale and standardize the data
- run the LDA to reduce dimensionality
- select a balanced number of random stocks (735 + 735)
- separate test and train dataframes
- run the three best algorithms (RF, SVM, KNN) and print the metrics
- run the best ensemble method (Voting Classifier with RF, SVM, MLP) and print the metrics 
- use the previous ensemble method to compare one by one the forecast and the actual outputs and the increase % for
100 stocks

'''

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import random
from numpy import set_printoptions
from progressbar import progressbar
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier
from itertools import combinations
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split


formatter = "{0:.3f}"

# uncomment to show every output rows
pd.set_option('display.max_rows', 100)

# uncomment to show every output columns
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 100)

# how the floating numbers are shown in the numpy arrays
set_printoptions(precision=3) 

# how the floating numbers are shown in pandas
pd.options.display.float_format = '{:,.3f}'.format  

# choose one of the three classes, the number of tickers to test (out of 1555) and the seed
#class_to_demo, increase = "Class 1", "Increase after 1y"
#class_to_demo, increase = "Class 2", "Increase after 3y"
class_to_demo, increase = "Class 3", "Increase after 5y"
tickers_to_test = 100
seed = 123

np.random.seed(seed)


ModuleNotFoundError: No module named 'progressbar'

In [52]:
''' Functions '''

def run_lda_with_three_targets(df):
    '''
    Run the LDA with a multiindex dataframe having three targets. 
    Create a df with a discriminant for each target and the 3 Ys
    
    :param df: a pandas dataframe
    :param comps: the number of components for PCA or the % of variance explained
    :return: the explained variance and a pandas dataframe with the PCA components    
    
    '''
    # split features and targets, extract column names and indexes
    col_names = df.columns.values

    x1 = df[col_names[0:-3]]
    Y1 = df[col_names[-3]].rename("Class 1")

    # this will be used for the 3 LDAs
    ind = x1.index.values

    x2 = df[col_names[0:-2]]
    Y2 = df[col_names[-2]].rename("Class 2")
    
    x3 = df[col_names[0:-1]]
    Y3 = df[col_names[-1]].rename("Class 3")
    
    # fit and transform
    clf = LinearDiscriminantAnalysis()
    linear_discriminants1 = clf.fit_transform(x1, Y1)
    linear_discriminants2 = clf.fit_transform(x2, Y2)
    linear_discriminants3 = clf.fit_transform(x3, Y3)

    # create 3 separate df with the linear discriminants
    lda_df1 = pd.DataFrame(data=linear_discriminants1, index=ind, columns=['LDA1'])
    lda_df2 = pd.DataFrame(data=linear_discriminants2, index=ind, columns=['LDA2'])
    lda_df3 = pd.DataFrame(data=linear_discriminants3, index=ind, columns=['LDA3'])
    
    # concatenate the 3 dfs
    lda_final_df = pd.concat([lda_df1, lda_df2], axis=1)
    lda_final_df = pd.concat([lda_final_df, lda_df3], axis=1)

    # concatenate the 3 Ys
    lda_final_df = pd.concat([lda_final_df, Y1], axis=1)
    lda_final_df = pd.concat([lda_final_df, Y2], axis=1)
    lda_final_df = pd.concat([lda_final_df, Y3], axis=1)
    
    
    return lda_final_df

def concat_two_df_along_columns(df1, df2):
    """
    Concat two dataframes on axis=1 (columns)
    :param df1: dataframe 1
    :param df2: dataframe 2
    :return: a new dataframe
    """
    df_concatenated = pd.concat([df1, df2], axis=1)

    return df_concatenated

def standardize_df(X):
    standardizer = StandardScaler().fit(X)
    standardizedX = standardizer.transform(X)

    return standardizedX

def scale_df(X):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaledX = scaler.fit_transform(X)

    return scaledX

def create_multiindex_dataframe(filename, index_list):
    '''
    Create a multiindex dataframe from a csv or xlsx file
    :param filename: a csv or xlsx file
    :param index_list: number of column levels, usually [0, 1]
    :return:
    '''

    if "xlsx" in filename:
        df = pd.read_excel(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)
    elif "csv" in filename:
        df = pd.read_csv(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)

    return df


In [53]:
''' Variables '''

index_list = [0, 1]

quarters_all = ["t0", "t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t10", "t11", "t12", "t13", "t14", "t15",
               "t16", "t17", "t18", "t19", "t20", "t21", "t22", "t23", "t24", "t25", "t26", "t27", "t28", "t29", "t30",
               "t31", "t32", "t33", "t34", "t35", "t36", "t37", "t38", "t39", "t40", "t41", "t42", "t43", "t44", "t45",
               "t46", "t47", "t48", "t49", "t50", "t51", "t52", "t53", "t54", "t55", "t56", "t57", "t58", "t59"]

features = ['adjusted_close', 'volume', 'totalRevenue', 'totalLiab', 'totalAssets', 'otherAssets', 
            'totalStockholderEquity', 'capitalExpenditures', 'incomeBeforeTax', 'researchDevelopment', 
            'incomeTaxExpense', 'netIncome', 'propertyPlantEquipment', 'netIncomeApplicableToCommonShares', 
            'sellingGeneralAdministrative', 'costOfRevenue', 'grossProfit', 'accountsPayable', 
            'operatingIncome', 'interestExpense', 'commonStock']


In [54]:
# load the data from the csv file and create a multiindex dataframe
filename = "dataset_with_classes.csv"
df_with_classes = create_multiindex_dataframe(filename, index_list)

# drop the tickers having 0s 
df_with_classes = df_with_classes.drop(['A', 'AAPL', 'AAWW'])

# check that there are no nulls and show the df shape
df_with_classes.isna().sum().sum(), df_with_classes.shape


(0, (1555, 849))

In [55]:
# the 40 quarters to train (10 years)
quarters_to_train = quarters_all[0:40]

# separate features and the 3 classes
X = df_with_classes.loc[:, (quarters_to_train, features)]
Y1 = df_with_classes.loc[:, ( "Output t43", "Class 1" )]
Y2 = df_with_classes.loc[:, ( "Output t51", "Class 2" )]
Y3 = df_with_classes.loc[:, ( "Output t59", "Class 3" )]

tickers = X.index.values.tolist()

# store the percentage increase for each class
Y1_increase = df_with_classes.loc[:, ( "Output t43", "Increase after 1 year" )].rename("Increase after 1y").to_frame()
Y2_increase = df_with_classes.loc[:, ( "Output t51", "Increase after 3 year" )].rename("Increase after 3y").to_frame()
Y3_increase = df_with_classes.loc[:, ( "Output t59", "Increase after 5 year" )].rename("Increase after 5y").to_frame()

# fit and transform data
scaled_df = scale_df(X)
standardized_df = standardize_df(scaled_df)

# recreate the dataframe with X
X = pd.DataFrame(data=standardized_df,
                 index=tickers,
                 columns=pd.MultiIndex.from_product([quarters_to_train, features]))

# concatenate the classes along the columns
Y = concat_two_df_along_columns(Y1, Y2)
Y = concat_two_df_along_columns(Y, Y3)

# concatenate X and Y
df_full = concat_two_df_along_columns(X, Y)


In [56]:
# run the LDA for each of the 3 classes and create a dataframe with 3 features 
df_full_lda = run_lda_with_three_targets(df_full)

# concatenate the increase
df_full_lda = concat_two_df_along_columns(df_full_lda, Y1_increase)
df_full_lda = concat_two_df_along_columns(df_full_lda, Y2_increase)
df_full_lda = concat_two_df_along_columns(df_full_lda, Y3_increase)

columns_lda = df_full_lda.columns.values # lda df column names

df_full_lda.head()

,LDA1,LDA2,LDA3,Class 1,Class 2,Class 3,Increase after 1y,Increase after 3y,Increase after 5y
AAN,3.125,1.642,-0.933,1,1,0,35.750,101.209,88.857
AAP,0.685,3.083,1.625,1,1,1,65.095,87.937,280.402
AAXN,-1.331,0.100,3.107,0,1,1,-1.689,74.046,352.713
AB,-1.324,-2.250,-3.092,0,0,0,-2.385,-23.909,35.378
ABB,-1.812,-2.101,-2.133,0,0,0,12.391,1.495,14.277


In [57]:
# create the conditions to separate stocks
condition_1 = df_full_lda.loc[:, class_to_demo] == 1
condition_0 = df_full_lda.loc[:, class_to_demo] == 0

# from 1555 total tickers, 735 have class label 1 and 820 class label 0 
tickers_1 = df_full_lda.index[condition_1].values.tolist()
tickers_0 = df_full_lda.index[condition_0].values.tolist()

# to balance the class, class label 0 must be 735 as well
random.seed(a=seed)

# select 735 random number of stocks having class label 0
tickers_0 = random.sample(tickers_0, k=735)

x_0 = df_full_lda.loc[tickers_0, ["LDA1", "LDA2", "LDA3"]]
x_1 = df_full_lda.loc[tickers_1, ["LDA1", "LDA2", "LDA3"]]

# concatenate and sort
x = pd.concat([x_0, x_1], axis=0)
x = x.sort_index(axis=0)

# same for Y
y_0 = df_full_lda.loc[tickers_0, class_to_demo]
y_1 = df_full_lda.loc[tickers_1, class_to_demo]
y = pd.concat([y_0, y_1], axis=0).to_frame()
y = y.sort_index(axis=0)

# same for the increase % 
Y3_increase_0 = df_full_lda.loc[tickers_0, increase]
Y3_increase_1 = df_full_lda.loc[tickers_1, increase]
Y3_increase = pd.concat([Y3_increase_0, Y3_increase_1], axis=0).to_frame()
Y3_increase = Y3_increase.sort_index(axis=0)

# create x and Y for test and train, the parameter stratify will select a balanced number of class labels
x_train, x_test, Y_train, Y_test = train_test_split(x, y, test_size=tickers_to_test, random_state=seed, stratify=y)


In [58]:
## Random Forests

randf = RandomForestClassifier(n_estimators=50, random_state=seed)

randf.fit(x_train, Y_train)
yhat = randf.predict(x_test)

acc_result = accuracy_score(Y_test, yhat)
prec_result = precision_score(Y_test, yhat)
rec_result = recall_score(Y_test, yhat)
f1_result = f1_score(Y_test, yhat)

print('RF - ' + 'Accuracy: %.3f' % acc_result, 'Precision: %.3f' % prec_result, 'Recall: %.3f' % rec_result, 'F1: %.3f' % f1_result)


# Support Vector Machine
svm_mod = svm.SVC(kernel='rbf', random_state=seed)

svm_mod.fit(x_train, Y_train)
yhat = svm_mod.predict(x_test)

acc_result = accuracy_score(Y_test, yhat)
prec_result = precision_score(Y_test, yhat)
rec_result = recall_score(Y_test, yhat)
f1_result = f1_score(Y_test, yhat)
    
print('SVM - ' + 'Accuracy: %.3f' % acc_result, 'Precision: %.3f' % prec_result, 'Recall: %.3f' % rec_result, 'F1: %.3f' % f1_result)


# k-Nearest Neighbors

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(x_train, Y_train)
yhat = knn.predict(x_test)

acc_result = accuracy_score(Y_test, yhat)
prec_result = precision_score(Y_test, yhat)
rec_result = recall_score(Y_test, yhat)
f1_result = f1_score(Y_test, yhat)

print('KNN - ' + 'Accuracy: %.3f' % acc_result, 'Precision: %.3f' % prec_result, 'Recall: %.3f' % rec_result, 'F1: %.3f' % f1_result)


RF - Accuracy: 0.950 Precision: 0.941 Recall: 0.960 F1: 0.950
SVM - Accuracy: 0.950 Precision: 0.941 Recall: 0.960 F1: 0.950
KNN - Accuracy: 0.950 Precision: 0.941 Recall: 0.960 F1: 0.950


In [59]:
# Ensemble Method (Voting Classifier)
# The best ensemble is (RF, SVM, MLP)

# initialize an MLP classifier
mlp = MLPClassifier(activation='relu', random_state=seed, hidden_layer_sizes=(5, 5), max_iter=500)

acc_result = 0
prec_result = 0
rec_result = 0
f1_result = 0

estimators=[("randf", randf), ("svm_mod", svm_mod), ("mlp", mlp)]
ensemble = VotingClassifier(estimators, voting="hard")

ensemble.fit(x_train, Y_train)
yhat = ensemble.predict(x_test)

acc_result = accuracy_score(Y_test, yhat)
prec_result = precision_score(Y_test, yhat)
rec_result = recall_score(Y_test, yhat)
f1_result = f1_score(Y_test, yhat)

print('Ensemble - ' + 'Accuracy: %.3f' % acc_result, 'Precision: %.3f' % prec_result, 'Recall: %.3f' % rec_result, 'F1: %.3f' % f1_result)


Ensemble - Accuracy: 0.950 Precision: 0.941 Recall: 0.960 F1: 0.950


In [60]:
# compare the forecast and the real output

stock_forecast = list(yhat)
best_stock = [] # stocks with label == 1

ticker_tested = Y_test.index.values.tolist()


for p in range(len(stock_forecast)):
    if stock_forecast[p] == 1:
        print(ticker_tested[p]  + " forecast: " + str(stock_forecast[p]) + " real: " + str(Y_test.loc[ticker_tested[p], class_to_demo]) + ' Increase ' + str(Y3_increase.loc[ticker_tested[p], increase]))
        best_stock.append(ticker_tested[p])
    if stock_forecast[p] == 0:
        print(ticker_tested[p]  + " forecast: " + str(stock_forecast[p]) + " real: " + str(Y_test.loc[ticker_tested[p], class_to_demo]) + ' Increase ' + str(Y3_increase.loc[ticker_tested[p], increase]))


PRU forecast: 1 real: 1 Increase 100.33
BMTC forecast: 1 real: 1 Increase 120.16
EQC forecast: 0 real: 0 Increase 21.888
XRX forecast: 0 real: 0 Increase 87.551
VTR forecast: 1 real: 1 Increase 107.344
BPOP forecast: 0 real: 0 Increase 27.699
BLL forecast: 1 real: 1 Increase 171.927
BSTC forecast: 0 real: 0 Increase 24.96
CWBC forecast: 0 real: 1 Increase 133.61
AAP forecast: 1 real: 1 Increase 280.402
MSFT forecast: 0 real: 0 Increase 86.838
FISV forecast: 1 real: 1 Increase 186.33
TECH forecast: 0 real: 0 Increase 49.676
ES forecast: 1 real: 1 Increase 453.922
MFA forecast: 1 real: 1 Increase 111.446
SLF forecast: 0 real: 0 Increase 62.639
SIGA forecast: 0 real: 0 Increase -78.9
AON forecast: 1 real: 1 Increase 140.667
LPTH forecast: 0 real: 0 Increase -33.843
TTC forecast: 1 real: 1 Increase 230.087
CUB forecast: 0 real: 0 Increase 37.978
KAMN forecast: 0 real: 0 Increase 95.849
IMKTA forecast: 1 real: 1 Increase 121.527
REV forecast: 1 real: 1 Increase 158.63
DENN forecast: 1 real: